In [1]:
import pandas as pd
import sklearn as sk
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# !pip3 install openpyxl # pandas dependancy for excel files



In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
try:
  df = pd.read_excel("./ML_Files/2_Clustering/EXL Shared Partner Data.xlsx")
except:
  df = pd.read_excel("/content/drive/Othercomputers/My MacBook Pro/Repositories/13.nagendra2025/ML_Files/2_Clustering/EXL Shared Partner Data.xlsx")


FileNotFoundError: [Errno 2] No such file or directory: './ML_Files/2_Clustering/EXL Shared Partner Data.xlsx'

In [ ]:
df.head()

In [ ]:
df.tail()

# Mindmap for solving clustering problem
## basic analysis -
## what is the dataset about
## how many
## what kind of - basic analysis on each kind
## any missing values - imputation
## categorical to numerical
## feature selection and extraction and further analysis
## basic plots
## finding optimum k -
## preprocessing - standardization
## clustering - elbow and silhouette

# some tricks
## after each step validate
## always use functions for so that you can reuse
## always code with giving commentary - speak and explain in english what i am doing while coding
## when writing functions, think in simple terms - like what are the inputs, what should be the output and how should i go from input to output -fill those two first and then figure out the logic in the middle afterwards
## like how keerti purswani does it - she always keeps it very simple and explanatory


In [ ]:
df.head()

In [ ]:
df.columns

In [ ]:
df.shape

In [ ]:
df.info()

df['Application_Nr'].value_counts()

In [ ]:
df['Application_Nr']

In [ ]:
df['Application_Nr'].value_counts().max()

In [ ]:
df.describe()

In [ ]:
df.isna().sum()

In [ ]:
df.columns

In [ ]:
from datetime import datetime
df['quotegenerated_year'] = df['QuoteGeneratedDate'].apply(lambda x: datetime.strftime(x,"%Y"))

In [ ]:
df['quotegenerated_year_month'] = df['QuoteGeneratedDate'].apply(lambda x: datetime.strftime(x,"%Y-%m"))

In [ ]:
df.head(2)

In [ ]:
df.info()

In [ ]:
df_yearly_offeredsum = df.groupby(["quotegenerated_year"])['OfferCoverageAmount'].sum()

In [ ]:
df_yearly_offeredsum.head()

In [ ]:
print(df_yearly_offeredsum.index)        # Index values: [2020, 2021, 2022]
print(df_yearly_offeredsum.index.name)   # Index name: 'quotegenerated_year'
print(df_yearly_offeredsum.name)         # Series name: 'OfferCoverageAmount'

In [ ]:

# Create a sample series with dummy data
dummy_series = pd.Series([10, 20, 30, 40, 50], name="dummy_data")
dummy_series.index.name = "dummy_index"
print(dummy_series)
print(dummy_series.index)
print(dummy_series.name)
print(dummy_series.index.name)



# Create a sample dataframe with dummy data
data = {
    'A': [1, 2, 3, 4, 5],
    'B': ['apple', 'banana', 'cherry', 'date', 'elderberry'],
    'C': [10.5, 20.3, 30.7, 40.2, 50.1]
}
df_sample = pd.DataFrame(data)
print(df_sample)



In [ ]:
df_yearly_offeredsum['2018']

In [ ]:
df_yearly_offeredsum.reset_index(name="coverage_sum")

In [ ]:
df_monthly_offeredsum = df.groupby(['quotegenerated_year_month'])['OfferCoverageAmount'].sum().reset_index(name='coverage_monthly_sum')

In [ ]:
df_monthly_offeredsum.head()

In [ ]:

sns.barplot(x=df_monthly_offeredsum['quotegenerated_year_month'], y=df_monthly_offeredsum['coverage_monthly_sum'])

In [ ]:
df.isna().sum()

In [ ]:
df = df.dropna(subset='Application_Nr')

In [ ]:
df.info()

In [ ]:
df.head(2)

In [ ]:
df = df[['OfferCoverageAmount','Offer_Stat_Desc', 'OfferPremiumAmount', 'OfferBillingMethod',
       'Pmnt_Freq_Desc','Offered_NAP','Issued_NAP', 'CurrentBillMethod', 'ProductType', 'Age_at_AppReceived',
       'PartnerMarker']]

In [ ]:
df.isna().sum()

In [ ]:
df.info()

In [ ]:
#data = df.dropna()
df.info()


In [ ]:
df.describe()

In [ ]:
def fill_null(data):
    print(data.isna().sum())
    con_cols = data.describe().columns
    print(type(con_cols))
    print(con_cols)
    cat_cols = [i for i in data.columns if i not in data.describe().columns]
    print(cat_cols)
    for i in con_cols:
        data[i].fillna(data[i].median(),inplace = True)
    for i in cat_cols:
        data[i].fillna(data[i].value_counts().index[0], inplace = True)
    print(data.isna().sum())

    return data, cat_cols, con_cols


In [ ]:
import warnings
warnings.filterwarnings('ignore')

cleaned_df, cat, con = fill_null(df)

In [ ]:
cat

In [ ]:
con

In [ ]:
cleaned_df.head(2)

In [ ]:
cleaned_df['OfferBillingMethod'].unique()

In [ ]:
# df_upd =  pd.get_dummies(df)
from sklearn.preprocessing import LabelEncoder

def label_encoding(data, cat):
    le = LabelEncoder()
    for i in cat:
        le.fit(data[i])
        data[i] = le.transform(data[i])

    return data

labelled_data = label_encoding(cleaned_df, cat)




In [ ]:
labelled_data.info()

In [ ]:
labelled_data.head(2)

# clustering

In [ ]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=8, random_state=10)

kmeans.fit(labelled_data)
print(kmeans.labels_)
labelled_data['cluster'] = kmeans.predict(labelled_data)

In [ ]:
labelled_data.head()

In [ ]:
labelled_data.tail()

In [ ]:
labels = kmeans.labels_
labels

In [ ]:
clus = pd.DataFrame(data=labels, columns=['clsters'])


In [ ]:
clus.head()

In [ ]:
clus.value_counts()

In [ ]:
labelled_data.head()

In [ ]:
labelled_data = labelled_data.drop(columns=['cluster'])

In [ ]:
from sklearn.preprocessing import StandardScaler

def preprocess(data):
    scaler = StandardScaler()
    # for i in data.columns:
    #     # Reshape the column to a 2D array for StandardScaler
    #     scaler.fit(data[[i]])
    #     data[i] = scaler.transform(data[[i]])
    scaler.fit(data)
    data = scaler.transform(data)
    return data

final_data = preprocess(labelled_data)

print(type(final_data))

In [ ]:
# final_data.head()
final_data

## WCSS - within cluster sum of squares - Elbow method

In [ ]:
wcss = []

for i in range(2,40):
    kmeans = KMeans(n_clusters=i, init='k-means++',random_state=10)
    kmeans.fit(final_data)
    wcss.append(kmeans.inertia_)

In [ ]:
wcss

In [ ]:
sns.set_theme()
plt.plot(range(2,40),wcss)
plt.xlabel("No of clusters")
plt.ylabel("WCSS_Value")
plt.title("Elbow point graph")
plt.show()

## Silhouette score - how well the clusters are formed

In [ ]:
from sklearn.metrics import silhouette_score

silhouette_scores = []

for i in range(2, 10):
    kmeans = KMeans(n_clusters=i, random_state=10)
    ss = silhouette_score(final_data, kmeans.fit_predict(final_data))
    silhouette_scores.append(ss)
    print(f"silhoutte score for #cluster:{i}: ", ss)
silhouette_scores

In [ ]:
sns.barplot(range(2,10),silhouette_scores)

# Final clustering with optimum k and visualization

In [ ]:
kmeans = KMeans(n_clusters=5, random_state=10)

kmeans.fit(final_data)
labels = kmeans.predict(final_data)

In [ ]:
x = final_data[:, [0, 2]]
x = x.values

y = labels

x_cetroids = kmeans.cluster_centers_[:,[0,2]]

In [ ]:
x

In [ ]:
y

In [ ]:
# plotting all the clusters and their Centroids

plt.figure(figsize=(8,8))
plt.scatter(X[Y==0,0], X[Y==0,1], s=50, c='green', label='Cluster 1')
plt.scatter(X[Y==1,0], X[Y==1,1], s=50, c='red', label='Cluster 2')
plt.scatter(X[Y==2,0], X[Y==2,1], s=50, c='yellow', label='Cluster 3')
plt.scatter(X[Y==3,0], X[Y==3,1], s=50, c='violet', label='Cluster 4')
plt.scatter(X[Y==4,0], X[Y==4,1], s=50, c='blue', label='Cluster 5')

# plot the centroids
plt.scatter(kmeans.cluster_centers_[:,0], kmeans.cluster_centers_[:,1], s=100, c='cyan', label='Centroids')

plt.title('Customer Groups')
plt.xlabel('OfferCoverageAmount')
plt.ylabel('OfferPremiumAmount')
plt.show()